In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# Question 1

Using minimize from scipy for MLE

In [2]:
df = pd.read_csv('project/data.csv') 

features = [
    'prop_starrating',
    'prop_review_score',
    'prop_brand_bool',
    'prop_location_score',
    'prop_accesibility_score',
    'prop_log_historical_price',
    'price_usd',
    'promotion_flag'
]

X_raw = df[features].values
y = df['booking_bool'].values
srch_ids = df['srch_id'].values

intercept_column = np.ones((X_raw.shape[0], 1))
X = np.hstack((intercept_column, X_raw))

#Search Group bins
unique_srch_ids, group_indices = np.unique(srch_ids, return_inverse=True)

In [3]:
# Optimizer
def neg_log_likelihood(beta):
    # Calculate Utility for every row
    V = np.dot(X, beta)
    
    #Weights
    exp_V = np.exp(V)
    
    # Calculate Denominators (sum of weights per search + 1.0 for Walk Away)
    sum_exp_V = np.bincount(group_indices, weights=exp_V)
    D = 1.0 + sum_exp_V
    
    # The Log-Likelihood Math Shortcut
    sum_log_D = np.sum(np.log(D))
    sum_yV = np.sum(y * V)
    
    # Return the negative score for Scipy to minimize
    return -(sum_yV - sum_log_D)

In [4]:
initial_beta = np.zeros(len(features) + 1) 

res = minimize(neg_log_likelihood, initial_beta, method='BFGS')

all_names = ['Intercept'] + features

beta_dict = dict(zip(all_names, res.x))

beta_df = pd.DataFrame({
    'Feature': all_names,
    'Coefficient': res.x
})

beta_df

,Feature,Coefficient
0,Intercept,-2.815326
1,prop_starrating,0.476125
2,prop_review_score,0.119897
3,prop_brand_bool,0.229923
4,prop_location_score,0.016338
5,prop_accesibility_score,0.562872
6,prop_log_historical_price,-0.037368
7,price_usd,-0.007323
8,promotion_flag,0.454029


### Model Coefficient Analysis

**The Baseline**
* `Intercept` (-2.815): High baseline friction. Users have a strong natural tendency to window-shop and walk away without booking unless the hotel features are highly compelling.

**The Highest Positive Drivers**
* `prop_accesibility_score` (+0.563): The strongest feature driver. Customers heavily prioritize convenience, location features, and physical accessibility.
* `prop_starrating` (+0.476): High impact. Customers clearly favor official, higher-tier quality ratings.
* `promotion_flag` (+0.454): Deal-hunting behavior is dominant. A visible promo badge is highly effective at overcoming the negative intercept and securing a booking.

**The Moderate Positive Drivers**
* `prop_brand_bool` (+0.230): Brand trust matters. Customers prefer recognized hotel chains over independent properties.
* `prop_review_score` (+0.120): Social proof works. Higher user reviews provide a solid, reliable boost to the hotel's utility.
* `prop_location_score` (+0.016): Minor positive impact. A good location helps, but it is vastly overshadowed by price, promos, and accessibility.

**The Negative Friction (Price & Value)**
* `price_usd` (-0.007): Standard economic friction. Higher absolute out-of-pocket costs directly reduce the probability of booking.
* `prop_log_historical_price` (-0.037): Luxury avoidance. Users actively shy away from properties that are known for being historically expensive, further proving the "deal hunter" persona.

# Question 2

### LP Formulation:
To bypass the non-linear fractional math of the MNL model, we need to flatten the problem into a Mixed-Integer Linear Program.

#### 1. Given Parameters
* $N$: The set of all candidate hotels.
* $p_j$: The price of hotel $j$.
* $v_j$: The predetermined MNL weight of hotel $j$ (calculated as $e^{V_j}$).

#### 2. Decision Variables
* $x_j \in \{0, 1\}$: Binary switch (1 if hotel $j$ is shown, 0 if hidden).
* $w_j \ge 0$: Continuous variable representing the probability a customer books hotel $j$.
* $w_0 \ge 0$: Continuous variable representing the probability the customer walks away.

#### 3. Objective Function
Maximize the Expected Revenue across the displayed assortment:
$$\max \sum_{j \in N} p_j w_j$$

#### 4. Constraints
**1. The Probability Axiom**
The total probability of all outcomes must equal 100%.
$$w_0 + \sum_{j \in N} w_j = 1.0$$

**2. The Hidden Rule**
A customer cannot buy a hotel that is not on the screen.
$$w_j \le x_j \quad \forall j \in N$$

**3. The MNL Ceiling (Upper Bound)**
The probability of a hotel cannot exceed its true MNL fraction.
$$w_j \le v_j w_0 \quad \forall j \in N$$

**4. The MNL Floor (Lower Bound)**
If a hotel is on the screen ($x_j = 1$), its probability must exactly equal its MNL fraction.
$$w_j \ge v_j w_0 - v_j(1 - x_j) \quad \forall j \in N$$

In [5]:
import gurobipy as gp
from gurobipy import GRB

In [6]:
def optimize_assortment(df, features, betas):
    # weight for the dataset
    X_raw = df[features].values
    intercept_column = np.ones((X_raw.shape[0], 1))
    X = np.hstack((intercept_column, X_raw))
    
    # set up
    V = np.dot(X, betas)
    weights = np.exp(V)
    prices = df['price_usd'].values
    num_hotels = len(df)
    hotel_indices = range(num_hotels)

    # Gurobi Model
    env = gp.Env(empty=True)
    env.setParam('OutputFlag', 0) 
    env.start()
    m = gp.Model("Assortment_Optimization", env=env)

    # decision variables
    x = m.addVars(hotel_indices, vtype=GRB.BINARY, name="x")
    w = m.addVars(hotel_indices, lb=0, name="w")
    w_0 = m.addVar(lb=0, name="w_0")

    # Objective: Maximize Expected Revenue
    m.setObjective(gp.quicksum(prices[j] * w[j] for j in hotel_indices), GRB.MAXIMIZE)

    # Constraints
    # Probabilities sum to 1
    m.addConstr(w_0 + gp.quicksum(w[j] for j in hotel_indices) == 1.0, "Sum_of_Probs")

    for j in hotel_indices:
        # Cannot buy what is hidden
        m.addConstr(w[j] <= x[j], f"Hidden_Rule_{j}")
        # Ceiling
        m.addConstr(w[j] <= weights[j] * w_0, f"MNL_Upper_{j}")
        # Floor
        m.addConstr(w[j] >= weights[j] * w_0 - weights[j] * (1 - x[j]), f"MNL_Lower_{j}")

    m.optimize()

    optimal_shelf = []
    for j in hotel_indices:
        if x[j].X > 0.5:
            optimal_shelf.append(j)
            
    expected_revenue = m.ObjVal
    walk_away_prob = w_0.X
    
    return optimal_shelf, expected_revenue, walk_away_prob

In [7]:
datasets = ['data1.csv', 'data2.csv', 'data3.csv', 'data4.csv']


for file in datasets:
    df_current = pd.read_csv(f'project/{file}')
    
    shelf, rev, walk_away = optimize_assortment(df_current, features, res.x)
    
    print(f"Results for {file}:")
    print(f"Optimal Hotels to Display (Indices): {shelf}")
    print(f"Total Hotels on Screen: {len(shelf)}")
    print(f"Probability of Customer Walking Away: {walk_away*100:.1f}%")
    print(f"Maximum Expected Revenue: ${rev:.2f}")
    print(" ")

Results for data1.csv:
Optimal Hotels to Display (Indices): [0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26]
Total Hotels on Screen: 18
Probability of Customer Walking Away: 20.8%
Maximum Expected Revenue: $107.35
 
Results for data2.csv:
Optimal Hotels to Display (Indices): [0, 1, 6, 7, 8, 9, 10, 21, 23, 25]
Total Hotels on Screen: 10
Probability of Customer Walking Away: 19.1%
Maximum Expected Revenue: $131.11
 
Results for data3.csv:
Optimal Hotels to Display (Indices): [0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24]
Total Hotels on Screen: 18
Probability of Customer Walking Away: 25.5%
Maximum Expected Revenue: $121.05
 
Results for data4.csv:
Optimal Hotels to Display (Indices): [3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26]
Total Hotels on Screen: 11
Probability of Customer Walking Away: 19.3%
Maximum Expected Revenue: $97.41
 


# Question 3

We maximize Expected Revenue where the purchase probability for each hotel follows the MNL model. Because prices appear in both the individual utilities and the shared denominator, we simplify the calculation by fixing the non-price attributes.

We decompose the utility $V_j$ into a constant part ($C_j$) and a price-dependent part. $C_j$ is pre-calculated using the $\beta$ coefficients from Problem 1 for all static features. This allows the optimizer to focus solely on the price variables:

$$\max_{\mathbf{p}} \sum_{j \in N} p_j \cdot \frac{e^{C_j + \beta_{price} \cdot p_j}}{1 + \sum_{k \in N} e^{C_k + \beta_{price} \cdot p_k}}$$

In [8]:
def optimize_dynamic_pricing(df, features, betas):
    # isolate price beta
    price_idx = features.index('price_usd')
    beta_price = betas[price_idx + 1] # +1 to skip intercept
    
    # Constant Utility 
    X_const = df[features].copy()
    X_const['price_usd'] = 0 
    X_const_mat = np.hstack([np.ones((len(X_const), 1)), X_const.values])
    C = np.dot(X_const_mat, betas)
    
    def objective(p):
        v = np.exp(C + beta_price * p)
        prob = v / (1 + np.sum(v))
        return -np.sum(p * prob)

    # p0 is the array of individual starting prices
    p0 = df['price_usd'].values
    
    res_pricing = minimize(objective, p0, method='L-BFGS-B', bounds=[(10, 1000)]*len(p0))
    return -res_pricing.fun, res_pricing.x

In [12]:
for file in datasets:
    df = pd.read_csv(f'project/{file}') 
    
    max_rev, opt_prices = optimize_dynamic_pricing(df, features, res.x)
    
    df['optimized_price'] = opt_prices
    
    print(f"{file}")
    print(f"Max Expected Revenue: ${max_rev:.2f}")
    
    display_cols = ['price_usd', 'optimized_price']
        
    pd.options.display.float_format = '{:,.2f}'.format
    display(df[display_cols])
    print("\n")

data1.csv
Max Expected Revenue: $177.81


,price_usd,optimized_price
0,150,314.37
1,140,314.36
2,145,314.36
3,125,314.35
4,154,314.36
5,144,314.36
6,131,314.36
7,91,314.40
8,77,314.26
9,107,314.36




data2.csv
Max Expected Revenue: $248.80


,price_usd,optimized_price
0,233,385.35
1,161,385.37
2,123,385.34
3,118,385.35
4,125,385.38
5,102,385.34
6,146,385.36
7,206,385.38
8,212,385.34
9,137,385.36




data3.csv
Max Expected Revenue: $176.47


,price_usd,optimized_price
0,140,313.02
1,211,313.00
2,150,313.00
3,144,313.02
4,191,313.02
5,195,313.01
6,115,313.00
7,281,313.02
8,128,313.01
9,101,312.91




data4.csv
Max Expected Revenue: $214.47


,price_usd,optimized_price
0,71,351.03
1,45,351.03
2,92,351.04
3,152,351.05
4,195,351.05
5,45,351.00
6,105,351.04
7,85,351.03
8,100,351.02
9,82,351.02



**Results Discussionn**

A counter-intuitive pattern emerged across all datasets: the optimal prices for all hotels within a single search query converged to a nearly identical uniform value (~$314 for Dataset 1, ~$385 for Dataset 2).

**Justification**

While pricing a 2-star and 5-star hotel identically defies real-world practices, it is the mathematical global optimum for an MNLmodel. Taking the first-order condition of the MNL expected revenue function yields the optimal price: 

$$p_i^* = \frac{-1}{\beta_{price}} + \text{Expected Revenue}$$

Because this equilibrium relies only on global price sensitivity and total expected revenue, completely ignoring product specific utilities like star rating. The solver dictates a uniform price ($p_1 = p_2 = \dots = p_n$). The optimizer eliminates "cheap" options to deliberately prevent low tier hotels from cannibalizing the probability share of higher tier hotels. This forces all non walk away customers into the highest-margin options, maximizing the total revenue pool.
